# Лабораторная работа №2

In [11]:
!pip install pyspark==3.5.0

In [2]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.12:0.17.0 pyspark-shell'

***1. Загрузка данных***

In [5]:
from pyspark.sql import SparkSession

# создаем сессию
spark = SparkSession.builder.appName("L2").getOrCreate()

path_xml = "/content/drive/MyDrive/Учёба/8 семестр/БД/posts_sample.xml"
path_csv = "/content/drive/MyDrive/Учёба/8 семестр/БД/programming-languages.csv"

languages_df = spark.read.csv(path_csv, header=True, inferSchema=True)
language_names = [row['name'].lower() for row in languages_df.dropna().collect()]

posts_df = spark.read.format("xml").option("rowTag", "row").load(path_xml)

posts_df.printSchema()

root
 |-- _AcceptedAnswerId: long (nullable = true)
 |-- _AnswerCount: long (nullable = true)
 |-- _Body: string (nullable = true)
 |-- _ClosedDate: timestamp (nullable = true)
 |-- _CommentCount: long (nullable = true)
 |-- _CommunityOwnedDate: timestamp (nullable = true)
 |-- _CreationDate: timestamp (nullable = true)
 |-- _FavoriteCount: long (nullable = true)
 |-- _Id: long (nullable = true)
 |-- _LastActivityDate: timestamp (nullable = true)
 |-- _LastEditDate: timestamp (nullable = true)
 |-- _LastEditorDisplayName: string (nullable = true)
 |-- _LastEditorUserId: long (nullable = true)
 |-- _OwnerDisplayName: string (nullable = true)
 |-- _OwnerUserId: long (nullable = true)
 |-- _ParentId: long (nullable = true)
 |-- _PostTypeId: long (nullable = true)
 |-- _Score: long (nullable = true)
 |-- _Tags: string (nullable = true)
 |-- _Title: string (nullable = true)
 |-- _ViewCount: long (nullable = true)



***2. Основная обработка данных***

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# выделяем год из даты и оставляем нужные столбцы
posts_filtered = posts_df.select(
    F.year("_CreationDate").alias("year"),
    F.col("_Tags").alias("tags")
).filter("year >= 2010 AND year <= 2020 AND tags IS NOT NULL")

# очищаем список имен языков (убираем лишние пробелы и переводим в нижний регистр)
clean_lang_list = [l.strip().lower() for l in language_names]

# функция для поиска языка программирования внутри тегов
def extract_language(tag_string):
    tag_string = tag_string.lower()
    for lang in clean_lang_list:
        if f"<{lang}>" in tag_string:
            return lang
    return None

# регистрируем функцию для работы в Spark
extract_lang_udf = F.udf(extract_language)

# применяем поиск и фильтруем посты, где язык не найден
posts_with_languages = posts_filtered.withColumn("language", extract_lang_udf(F.col("tags"))) \
                                     .filter(F.col("language").isNotNull())

# группируем по году и языку, считаем количество упоминаний
lang_usage_report = posts_with_languages.groupBy("year", "language").count()

***3. Топ 10 для каждого года***

In [7]:
# делим данные по годам и сортируем по количеству упоминаний внутри каждого года
window_spec = Window.partitionBy("year").orderBy(F.col("count").desc())

# нумеруем строки внутри каждого года и оставляем только первые 10
final_report = lang_usage_report.withColumn("rank", F.row_number().over(window_spec)) \
                                .filter(F.col("rank") <= 10) \
                                .select("year", "language", "count") \
                                .orderBy("year", F.desc("count"))

final_report.show(110)

+----+-----------+-----+
|year|   language|count|
+----+-----------+-----+
|2010|       java|   52|
|2010| javascript|   44|
|2010|        php|   42|
|2010|     python|   25|
|2010|objective-c|   23|
|2010|          c|   20|
|2010|       ruby|   11|
|2010|     delphi|    7|
|2010|applescript|    3|
|2010|          r|    3|
|2011|        php|   97|
|2011|       java|   92|
|2011| javascript|   82|
|2011|     python|   35|
|2011|objective-c|   33|
|2011|          c|   24|
|2011|       ruby|   17|
|2011|     delphi|    8|
|2011|       perl|    8|
|2011|       bash|    7|
|2012|        php|  136|
|2012| javascript|  129|
|2012|       java|  124|
|2012|     python|   65|
|2012|objective-c|   45|
|2012|          c|   27|
|2012|       ruby|   25|
|2012|       bash|    9|
|2012|          r|    9|
|2012|      scala|    6|
|2013| javascript|  196|
|2013|       java|  191|
|2013|        php|  173|
|2013|     python|   87|
|2013|objective-c|   40|
|2013|          c|   36|
|2013|       ruby|   30|


***4. Сохраняем в формат Parquet***

In [12]:
final_report.write.mode("overwrite").parquet("/content/drive/MyDrive/Учёба/8 семестр/БД/top10_languages_report.parquet")